# 01b - Pré-processamento dos pedidos rejeitados

Pré-processamento do ficheiro *rejected* (Fase 2 do enunciado, *Pre-processing*). Os pedidos rejeitados não entram na regressão nem na classificação. São usados em duas análises que comparam aceites e rejeitados:
- o **clustering de estados** (taxas de aprovação, montantes pedidos e *risk scores* por estado);
- os **testes de hipóteses** "um *risk score* mais baixo implica mais rejeição" e "o montante pedido difere entre aceites e rejeitados".

**Ordem dos passos** (a mesma do `01_preprocessamento.ipynb`): primeiro só o que não depende de nenhuma estatística dos dados, ou seja, a verificação estrutural e a **partição temporal** com os cortes da Fase 1. A partir daí, todas as análises usam **só o desenvolvimento**.

O enunciado pede três cuidados específicos com este ficheiro, tratados nas secções indicadas:
- `Risk_Score` = 0 conta como valor em falta;
- o DTI vem em texto com "%" (secção 4);
- há artefactos de introdução de dados (ex.: DTI de 4042.11%).

**Antes de correr:** é preciso ter corrido `00_conversao_dados.ipynb`. O ficheiro *rejected* ocupa cerca de 3,5 GB em memória.

In [1]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.exclusions import ExclusionLog, write_exclusions_markdown
from lending.parsing import parse_iso_date
from lending.partition import TemporalPartitioner
from lending.rejected import (DTI_CODES, DTI_MAX, NEW_PRODUCTS_POLICY_CODE, REJECTED_SCHEMA,
                              REJECTED_SCHEMA_RENAMED, REJECTED_UNITS, RejectedCleaner)
from lending.schema import SchemaReport
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Ficheiro de origem e verificação estrutural

No *accepted* havia linhas que não eram empréstimos (totais e separadores das exportações). Aqui verifica-se o mesmo: se as colunas são as 9 esperadas e se há linhas vazias ou com uma estrutura diferente.

In [2]:
rejected = PickleStore.load(config.REJECTED_RAW_PKL, expected_type=pd.DataFrame)
print(f"rejected: {len(rejected):,} linhas x {rejected.shape[1]} colunas")

assert list(rejected.columns) == list(REJECTED_SCHEMA), "colunas diferentes das esperadas"
print("Colunas iguais às do schema:", list(rejected.columns))

# Linhas sem nenhum valor e linhas sem data do pedido (não podiam ser atribuídas a uma partição)
print(f"Linhas totalmente vazias: {rejected.isna().all(axis=1).sum()}")
print(f"Linhas sem Application Date: {rejected['Application Date'].isna().sum()}")

rejected: 27,648,741 linhas x 9 colunas
Colunas iguais às do schema: ['Amount Requested', 'Application Date', 'Loan Title', 'Risk_Score', 'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length', 'Policy Code']
Linhas totalmente vazias: 0
Linhas sem Application Date: 0


Não há linhas estruturais: todas as linhas são pedidos, todas têm data. Não é preciso nenhuma remoção antes da partição.

## 2. Partição temporal

Os pedidos rejeitados seguem os **mesmos cortes** dos aceites (`desenho_split.md`), usando a data do pedido (`Application Date`):

| Partição | Período | Utilização |
|---|---|---|
| Antes do primeiro aceite | 2007-05 | Fora do estudo: não há empréstimos aceites neste mês, por isso não há com que comparar |
| Desenvolvimento | 2007-06 a 2014-12 | Clustering de estados, testes de hipóteses |
| Teste final | 2015 | Não usado: os rejeitados não entram nos modelos supervisionados |
| Reservado | 2016-2018 | Não usado na Parte 1 |

Converter a data é parsing determinístico (um formato fixo, `AAAA-MM-DD`), por isso pode ser feito antes da partição. Os cortes são meses fixos em `config.REJECTED_PARTITIONS`, nunca a ordem das linhas.

In [3]:
application_date = parse_iso_date(rejected["Application Date"])
print(f"Application Date: {application_date.min():%Y-%m-%d} a {application_date.max():%Y-%m-%d}")

partitioner = TemporalPartitioner(config.REJECTED_PARTITIONS)
partitioner.summary(application_date)

Application Date: 2007-05-26 a 2018-12-31


,start,end,rows,pct
partition,,,,
before_accepted,2007-05,2007-05,49,0.00
development,2007-06,2014-12,3450084,12.48
test,2015-01,2015-12,2859379,10.34
reserved,2016-01,2018-12,21339229,77.18


In [4]:
# Pedidos por ano e partição: mostra o crescimento dos pedidos depois de 2012
pd.crosstab(application_date.dt.year.rename("ano"),
            partitioner.assign(application_date).rename("partição"))

partição,before_accepted,development,test,reserved
ano,,,,
2007,49,5225,0,0
2008,0,25596,0,0
2009,0,56991,0,0
2010,0,112561,0,0
2011,0,217792,0,0
2012,0,337277,0,0
2013,0,760942,0,0
2014,0,1933700,0,0
2015,0,0,2859379,0


In [5]:
log = ExclusionLog("rejected -> desenvolvimento")
sizes = partitioner.summary(application_date)["rows"]

log.record(
    step="Pedidos anteriores ao primeiro aceite",
    kind="partição",
    rule="Application Date em 2007-05",
    reason="O primeiro empréstimo aceite é de 2007-06: sem aceites no mesmo mês, não há comparação",
    rows_before=len(rejected),
    rows_removed=int(sizes["before_accepted"]),
)
log.record(
    step="Separar o teste final",
    kind="partição",
    rule="Application Date de 2015-01 a 2015-12",
    reason="Mesmo corte dos aceites; os rejeitados do teste não são usados (não entram nos modelos)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["test"]),
)
log.record(
    step="Separar o período reservado",
    kind="partição",
    rule="Application Date de 2016-01 a 2018-12",
    reason="Não usado na Parte 1 (mesmo corte dos aceites)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["reserved"]),
)

# Só o desenvolvimento passa para memória; o ficheiro completo é libertado
rejected_dev = partitioner.split(rejected, application_date, names=["development"])["development"]
assert len(rejected_dev) == log.rows_after == sizes["development"]

del rejected, application_date
_ = gc.collect()

log.to_frame()

,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Pedidos anteriores ao primeiro aceite,partição,Application Date em 2007-05,"O primeiro empréstimo aceite é de 2007-06: sem aceites no mesmo mês, não há comparação",27648741,49,27648692
2,Separar o teste final,partição,Application Date de 2015-01 a 2015-12,Mesmo corte dos aceites; os rejeitados do teste não são usados (não entram nos modelos),27648692,2859379,24789313
3,Separar o período reservado,partição,Application Date de 2016-01 a 2018-12,Não usado na Parte 1 (mesmo corte dos aceites),24789313,21339229,3450084


## 3. Guardar a partição

Tal como no *accepted*, a partição é guardada **ainda em texto**, sem nenhum valor alterado:
- `rejected_dev_raw.pkl`: pedidos rejeitados do desenvolvimento;
- `rejected_dev_exclusions.pkl`: o registo de exclusões, que os passos seguintes continuam.

O teste e o período reservado não são guardados: não são usados e continuam disponíveis em `rejected_raw.pkl`.

In [6]:
PickleStore.save(rejected_dev, config.REJECTED_DEV_RAW_PKL)
PickleStore.save(log, config.REJECTED_DEV_LOG_PKL)
print(f"Desenvolvimento: {len(rejected_dev):,} linhas -> {config.REJECTED_DEV_RAW_PKL.name}")
print(f"Registo:         {log} -> {config.REJECTED_DEV_LOG_PKL.name}")

Desenvolvimento: 3,450,084 linhas -> rejected_dev_raw.pkl
Registo:         ExclusionLog('rejected -> desenvolvimento', 3 passos, 3450084 linhas) -> rejected_dev_exclusions.pkl


## 4. Conversão de tipos (parsing determinístico)

Cada coluna tem um tipo declarado à mão em `REJECTED_SCHEMA` (`src/lending/rejected.py`) e é convertida pelo mesmo `SchemaParser` do *accepted*:

| Coluna original | Nome usado | Tipo | Conversão |
|---|---|---|---|
| Amount Requested | `amount_requested` | number | "1000.0" -> 1000.0 |
| Application Date | `application_date` | date | "2007-05-26" -> data |
| Loan Title | `title` | text | espaços nas pontas retirados; só espaços -> em falta |
| Risk_Score | `risk_score` | number | "693.0" -> 693.0 |
| Debt-To-Income Ratio | `dti` | percent | "38.64%" -> 38.64 (escala 0–100, como o `dti` do *accepted*) |
| Zip Code | `zip_code` | category | "481xx" (3 primeiros dígitos, como no *accepted*) |
| State | `addr_state` | category | sigla do estado |
| Employment Length | `emp_length` | emp_length | "< 1 year" -> 0 … "10+ years" -> 10 (mesma tabela do *accepted*) |
| Policy Code | `policy_code` | number | "0.0" -> 0.0 |

As colunas passam a ter o **nome da coluna equivalente do *accepted***, para que comparar aceites e rejeitados seja direto. Um valor com formato inesperado dá erro em vez de passar a valor em falta sem ninguém reparar.

In [7]:
cleaner = RejectedCleaner()
rejected_dev_parsed = cleaner.parse(rejected_dev)

# Controlo: só texto vazio pode passar a valor em falta (o parser dá erro noutros casos)
cleaner.summary(rejected_dev, rejected_dev_parsed)

,kind,dtype,filled_before,filled_after,blank_to_missing
amount_requested,number,float64,3450084,3450084,0
application_date,date,datetime64[ns],3450084,3450084,0
title,text,str,3450070,3434831,15239
risk_score,number,float64,3147810,3147810,0
dti,percent,float64,3450084,3450084,0
zip_code,category,str,3450061,3450061,0
addr_state,category,str,3450063,3450063,0
emp_length,emp_length,float64,3399024,3399024,0
policy_code,number,float64,3450084,3450084,0


In [8]:
# Antes e depois, nas mesmas linhas (amostra aleatória com semente, só para ver a conversão)
sample = rejected_dev.sample(5, random_state=config.RANDOM_SEED).index
display(rejected_dev.loc[sample])
display(rejected_dev_parsed.loc[sample])

,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
54717,10000.0,2009-06-05,I do not expect a loan.....,647.0,3.05%,894xx,NV,1 year,0.0
8080600,10000.0,2014-03-31,debt_consolidation,644.0,20.03%,553xx,MN,< 1 year,0.0
7795486,15000.0,2014-01-14,debt_consolidation,653.0,10.02%,640xx,MO,< 1 year,0.0
7995174,6000.0,2014-03-11,moving,501.0,29.22%,490xx,MI,< 1 year,0.0
7390037,6000.0,2013-08-09,small_business,558.0,68.4%,279xx,NC,< 1 year,0.0


,amount_requested,application_date,title,risk_score,dti,zip_code,addr_state,emp_length,policy_code
54717,10000.0,2009-06-05,I do not expect a loan.....,647.0,3.05,894xx,NV,1.0,0.0
8080600,10000.0,2014-03-31,debt_consolidation,644.0,20.03,553xx,MN,0.0,0.0
7795486,15000.0,2014-01-14,debt_consolidation,653.0,10.02,640xx,MO,0.0,0.0
7995174,6000.0,2014-03-11,moving,501.0,29.22,490xx,MI,0.0,0.0
7390037,6000.0,2013-08-09,small_business,558.0,68.40,279xx,NC,0.0,0.0


In [9]:
# Tabelas de conversão: cada texto corresponde a um só número
emp_check = (pd.DataFrame({"texto": rejected_dev["Employment Length"],
                           "anos": rejected_dev_parsed["emp_length"]})
             .value_counts(dropna=False).rename("pedidos").reset_index()
             .sort_values("anos", na_position="last"))
assert emp_check["texto"].dropna().is_unique
display(emp_check)

# DTI: todos os valores preenchidos terminavam em "%" e passaram a número
print(f"DTI: {rejected_dev['Debt-To-Income Ratio'].notna().sum():,} preenchidos -> "
      f"{rejected_dev_parsed['dti'].notna().sum():,} números")

,texto,anos,pedidos
0,< 1 year,0.0,2923325
5,1 year,1.0,40763
4,2 years,2.0,44895
6,3 years,3.0,36670
7,4 years,4.0,27747
1,5 years,5.0,144348
8,6 years,6.0,21285
9,7 years,7.0,18009
10,8 years,8.0,16076
11,9 years,9.0,12702


DTI: 3,450,084 preenchidos -> 3,450,084 números


In [10]:
# A partir daqui trabalha-se com o desenvolvimento já convertido
# (a versão em texto continua guardada em rejected_dev_raw.pkl)
rejected_dev = rejected_dev_parsed
del rejected_dev_parsed
_ = gc.collect()

rejected_dev.dtypes.rename("dtype").to_frame()

,dtype
amount_requested,float64
application_date,datetime64[ns]
title,str
risk_score,float64
dti,float64
zip_code,str
addr_state,str
emp_length,float64
policy_code,float64


## 5. Schema e unidades

O `SchemaReport` (o mesmo do *accepted*) descreve cada coluna: tipo, unidade, intervalo observado e categorias mais frequentes. A tabela fica em `docs/02_preprocessamento/schema_rejected.md`.

In [11]:
schema_report = SchemaReport(schema=REJECTED_SCHEMA_RENAMED, units=REJECTED_UNITS)
schema_description = schema_report.describe(rejected_dev)

out = schema_report.write_markdown(schema_description,
                                   config.DOCS_DIR / "02_preprocessamento" / "schema_rejected.md",
                                   dataset="rejected, desenvolvimento", n_rows=len(rejected_dev),
                                   notebook="01b_rejeitados.ipynb")
print("Escrito:", out.relative_to(config.PROJECT_ROOT))
schema_description

Escrito: docs\02_preprocessamento\schema_rejected.md


,kind,unit,dtype,min,max,distinct,examples
column,,,,,,,
amount_requested,number,USD,float64,0,"1,400,000",2085,
application_date,date,data (dia),datetime64[ns],2007-06,2014-12,2771,
title,text,texto,str,,,72325,
risk_score,number,pontos,float64,0,990,688,
dti,percent,%,float64,-1,"50,000,031.49",47445,
zip_code,category,categoria,str,,,998,"112xx, 300xx, 770xx"
addr_state,category,categoria,str,,,51,"CA, TX, NY"
emp_length,emp_length,anos,float64,0,10,11,
policy_code,number,código,float64,0,2,2,


## 6. Valores em falta

Como no *accepted*, a pergunta não é só "quanto falta?" mas "porquê falta?". A tabela mostra a % em falta por ano do pedido, já com os tipos convertidos e antes de qualquer correção.

In [12]:
year = rejected_dev["application_date"].dt.year.rename("ano")
(rejected_dev.isna().groupby(year).mean() * 100).round(2)

,amount_requested,application_date,title,risk_score,dti,zip_code,addr_state,emp_length,policy_code
ano,,,,,,,,,
2007,0.0,0.0,0.00,1.99,0.0,0.00,0.00,0.00,0.0
2008,0.0,0.0,7.15,9.79,0.0,0.08,0.08,0.00,0.0
2009,0.0,0.0,17.82,10.92,0.0,0.00,0.00,0.02,0.0
2010,0.0,0.0,2.90,7.09,0.0,0.00,0.00,2.32,0.0
2011,0.0,0.0,0.00,1.28,0.0,0.00,0.00,0.88,0.0
2012,0.0,0.0,0.00,1.29,0.0,0.00,0.00,1.06,0.0
2013,0.0,0.0,0.00,3.13,0.0,0.00,0.00,1.47,0.0
2014,0.0,0.0,0.00,13.16,0.0,0.00,0.00,1.64,0.0


- `risk_score` falta em 13% dos pedidos de 2014. Nos anos anteriores falta pouco, mas há outro problema: zeros (secção 7).
- `title` (texto livre) falta em 2008–2010, até 18% em 2009. Não é usado nas análises.
- `emp_length` falta em 1–2% a partir de 2010; `zip_code` e `addr_state` quase nunca faltam (21 pedidos sem estado).

## 7. *Risk score*

### 7.1 Zeros

O enunciado diz que `Risk_Score` = 0 conta como valor em falta: não é uma pontuação possível (o FICO vai de 300 a 850).

In [13]:
risk_zero = rejected_dev["risk_score"] == 0
pd.DataFrame({"risk_score = 0": risk_zero.groupby(year).sum(),
              "em falta (NaN)": rejected_dev["risk_score"].isna().groupby(year).sum()})

,risk_score = 0,em falta (NaN)
ano,,
2007,221,104
2008,1028,2506
2009,3324,6226
2010,9525,7977
2011,19655,2778
2012,18759,4338
2013,34240,23822
2014,0,254523


Os zeros existem até 2013 e desaparecem em 2014, quando os valores em falta passam a vir vazios. É o mesmo significado guardado de duas formas: **regra:** `risk_score` = 0 -> valor em falta.

### 7.2 Mudança de escala em novembro de 2013

O `risk_score` chega a 990, acima do máximo do FICO (850). A célula seguinte mostra o mínimo e o máximo de cada dia à volta da mudança (sem os zeros).

In [14]:
positive = rejected_dev["risk_score"] > 0
day = rejected_dev["application_date"]
window = (day >= "2013-10-30") & (day <= "2013-11-10") & positive
display(rejected_dev.loc[window].groupby("application_date")["risk_score"]
        .agg(["min", "max", "count"]))

# Por ano: pedidos com risk_score acima de 850 (impossível em FICO)
(rejected_dev["risk_score"] > 850).groupby(year).sum().rename("risk_score > 850").to_frame()

,min,max,count
application_date,,,
2013-10-30,433.0,834.0,3642
2013-10-31,422.0,836.0,2858
2013-11-01,435.0,828.0,2805
2013-11-02,445.0,844.0,2042
2013-11-03,438.0,835.0,1848
2013-11-04,445.0,834.0,3199
2013-11-05,435.0,834.0,3338
2013-11-06,501.0,980.0,452
2013-11-07,501.0,971.0,1425


,risk_score > 850
ano,
2007,0
2008,0
2009,0
2010,0
2011,0
2012,0
2013,578
2014,5537


Até 2013-11-05 o `risk_score` vai de cerca de 420 a 850, como o FICO. **A partir de 2013-11-06** o mínimo passa a 501 e o máximo chega a 990, o intervalo do VantageScore (outra pontuação de crédito). A mudança é num dia exato, por isso não é uma mudança nos requerentes: é a LendingClub a mudar a pontuação que regista.

**Consequência:** um `risk_score` de 700 não quer dizer o mesmo antes e depois desta data. Não há uma conversão exata entre as duas escalas, por isso os valores não são alterados. Acrescenta-se uma coluna `risk_score_scale` ("FICO" / "VantageScore"), definida só pela data (`VANTAGE_START`).

⚠️ **Para os testes de hipóteses e o clustering:** comparar o `risk_score` dos rejeitados com o FICO dos aceites (`fico_range_low`/`fico_range_high`) só é válido no período FICO (até 2013-11-05). Misturar as duas escalas numa média ou num cluster daria resultados sem significado.

## 8. DTI

O enunciado avisa que o DTI vem em texto com "%" (já convertido na secção 4) e que há artefactos de introdução, dando o exemplo de 4042.11%. Nos aceites, o DTI do desenvolvimento vai de 0 a 39,99%.

In [15]:
dti = rejected_dev["dti"]
print(f"DTI: mínimo {dti.min():g}, mediana {dti.median():g}, máximo {dti.max():,.2f}")

# Valores repetidos que não parecem rácios: -1 e números redondos muito altos
print("Valores mais frequentes fora de 0-100:")
display(dti[(dti < 0) | (dti > 100)].value_counts().head(8).rename("pedidos").to_frame())

# O -1 aparece quase sempre quando também falta o risk_score: é um código de "sem dados"
no_risk = rejected_dev["risk_score"].isna() | risk_zero
pd.crosstab(dti.eq(-1).rename("dti = -1"), no_risk.rename("sem risk_score"))

DTI: mínimo -1, mediana 16.36, máximo 50,000,031.49
Valores mais frequentes fora de 0-100:


,pedidos
dti,
-1.00,265406
9999.00,6723
99999.00,4399
199998.00,1391
120.00,70
329.08,56
150.00,53
108.00,52


sem risk_score,False,True
dti = -1,,
False,3053709,130969
True,7349,258057


In [16]:
# Fora dos códigos, a distribuição dos valores acima de 100% (sem um corte natural)
dti_code = dti.isin(DTI_CODES)
above_100 = dti[(dti > 100) & ~dti_code]
print(f"Acima de 100% (sem os códigos): {len(above_100):,} pedidos "
      f"({100 * len(above_100) / len(dti):.2f}%) | 4042.11% presente: {(dti == 4042.11).sum()}")
pd.cut(above_100, [100, 200, 500, DTI_MAX, 10_000, 100_000, dti.max()]).value_counts(sort=False).rename("pedidos").to_frame()

Acima de 100% (sem os códigos): 80,802 pedidos (2.34%) | 4042.11% presente: 1


,pedidos
dti,
"(100.0, 200.0]",36841
"(200.0, 500.0]",26967
"(500.0, 1000.0]",8648
"(1000.0, 10000.0]",5338
"(10000.0, 100000.0]",2676
"(100000.0, 50000031.49]",332


**Decisões:**

| Valores | Pedidos | Interpretação | Regra |
|---|---:|---|---|
| −1 | 265 406 | Código de "sem dados": 97% destes pedidos também não têm `risk_score` | -> valor em falta |
| 9 999, 99 999, 199 998 | 12 513 | Números redondos repetidos milhares de vezes: valores de teto do campo, não rácios | -> valor em falta |
| Acima de 1000% | ≈ 8 300 | As dívidas mensais valeriam mais de 10 vezes o rendimento. Quase sempre é um rendimento mal introduzido (ex.: um valor mensal no campo anual). Inclui o 4042.11% do enunciado | -> valor em falta |
| 100% a 1000% | ≈ 72 000 | Plausível num pedido rejeitado: dívidas acima do rendimento são precisamente um motivo de rejeição | Fica |
| 0% | ≈ 203 000 | Possível (sem outras dívidas) | Fica |

O corte em 1000% é uma escolha: não há um corte natural na distribuição. Por isso:
- afeta só ≈ 0,24% dos pedidos;
- as análises do DTI dos rejeitados devem usar estatísticas robustas (mediana, testes de postos), que quase não dependem da posição exata do corte.

## 9. Montante pedido

Nos aceites do desenvolvimento, `loan_amnt` vai de 500 a 35 000 USD (o máximo da plataforma até 2016).

In [17]:
amount = rejected_dev["amount_requested"]
print(f"Montante pedido: mínimo {amount.min():g}, mediana {amount.median():,.0f}, máximo {amount.max():,.0f}")
pd.DataFrame({"= 0": (amount == 0).groupby(year).sum(),
              "< 1 000": (amount < 1000).groupby(year).sum(),
              "> 35 000": (amount > 35_000).groupby(year).sum()})

Montante pedido: mínimo 0, mediana 10,000, máximo 1,400,000


,= 0,< 1 000,> 35 000
ano,,,
2007,0,230,0
2008,0,195,1
2009,0,0,1
2010,0,0,24
2011,0,0,217
2012,2,2,169
2013,0,0,8
2014,0,0,17291


- **0 USD** (2 pedidos): um pedido sem montante não é um valor real -> valor em falta.
- **Acima de 35 000 USD** (17 711 pedidos, quase todos de 2014): são pedidos reais acima do limite da plataforma, provavelmente uma das razões da rejeição. **Ficam.** Na hipótese "o montante pedido difere entre aceites e rejeitados", parte da diferença vem desta regra da plataforma, e isso tem de ser dito na interpretação.

## 10. Anos de emprego

A % de pedidos com "< 1 year" (0 anos) muda muito ao longo do tempo:

In [18]:
(rejected_dev["emp_length"] == 0).groupby(year).mean().mul(100).round(1).rename("% com 0 anos").to_frame()

,% com 0 anos
ano,
2007,27.0
2008,26.1
2009,29.0
2010,66.5
2011,83.5
2012,86.8
2013,85.6
2014,87.8


Passa de cerca de 27% em 2007–2009 para 85% a partir de 2011. Uma mudança tão grande e tão rápida não é uma mudança nos requerentes: deve ser o valor por omissão do formulário ou de um campo que deixou de ser pedido. Não há forma de separar quem tinha mesmo menos de um ano de emprego. **Fica como está e documentado:** o `emp_length` dos rejeitados não é fiável a partir de 2010 e não deve ser usado nas comparações com os aceites.

## 11. Código de política

No *accepted*, todos os empréstimos do desenvolvimento têm `policy_code` = 1 (produtos públicos). O dicionário de dados diz que `policy_code` = 2 são *"new products not publicly available"*.

In [19]:
pd.crosstab(year, rejected_dev["policy_code"].rename("policy_code"))

policy_code,0.0,2.0
ano,,
2007,5225,0
2008,25596,0
2009,56991,0
2010,112561,0
2011,217792,0
2012,337277,0
2013,760942,0
2014,1927459,6241


Os rejeitados têm 0 (o código dos pedidos dos produtos públicos neste ficheiro) e, em 2014, 6 241 pedidos com 2. **Regra: excluir `policy_code` = 2.** Os aceites com que se comparam são todos de produtos públicos; manter estes pedidos misturaria produtos diferentes na taxa de aprovação.

## 12. Pedidos repetidos

O ficheiro não tem identificador do pedido. Procuram-se linhas iguais em **todas** as colunas, incluindo o dia, o título escrito pelo requerente, o código postal, o montante, o *risk score* e o DTI.

In [20]:
public = rejected_dev[rejected_dev["policy_code"] != NEW_PRODUCTS_POLICY_CODE]
repeated = public.duplicated(keep="first")
print(f"Linhas iguais a uma anterior: {repeated.sum():,} ({100 * repeated.mean():.2f}%)")

# Exemplos (amostra aleatória com semente): o grupo completo de cada linha repetida
example_keys = public[repeated].sample(3, random_state=config.RANDOM_SEED)
public.merge(example_keys.drop_duplicates(), how="inner").head(10)

Linhas iguais a uma anterior: 19,615 (0.57%)


,amount_requested,application_date,title,risk_score,dti,zip_code,addr_state,emp_length,policy_code
0,3000.0,2014-04-08,credit_card,NaN,-1.0,941xx,CA,0.0,0.0
1,3000.0,2014-04-08,credit_card,NaN,-1.0,941xx,CA,0.0,0.0
2,3000.0,2014-04-08,credit_card,NaN,-1.0,941xx,CA,0.0,0.0
3,5000.0,2014-06-26,other,NaN,-1.0,125xx,NY,0.0,0.0
4,5000.0,2014-06-26,other,NaN,-1.0,125xx,NY,0.0,0.0
5,10000.0,2014-09-03,debt_consolidation,NaN,-1.0,701xx,LA,0.0,0.0
6,10000.0,2014-09-03,debt_consolidation,NaN,-1.0,701xx,LA,0.0,0.0


Com todas as colunas iguais no mesmo dia, incluindo o texto livre do título, é quase certo que se trata do **mesmo pedido submetido mais de uma vez**. Deixá-los contaria várias rejeições onde houve uma, e inflacionaria a taxa de rejeição do estado. **Regra: fica a primeira linha de cada grupo.**

Limitação: dois requerentes diferentes com exatamente os mesmos valores no mesmo dia seriam juntados. É muito improvável, porque teriam de coincidir em todas as 9 colunas.

## 13. Aplicar a limpeza e guardar

As regras das secções 7 a 12 estão na classe `RejectedCleaner` (`src/lending/rejected.py`), aplicadas por esta ordem ao desenvolvimento em texto:
1. conversão de tipos (secção 4);
2. exclusões, sobre os valores originais: `policy_code` = 2; pedidos repetidos;
3. correções (valor -> em falta): `risk_score` = 0; DTI = −1, 9 999, 99 999 ou 199 998; DTI > 1000%; montante = 0;
4. coluna `risk_score_scale`.

As exclusões vêm antes das correções porque as correções juntam valores diferentes em "em falta" (ex.: DTI −1 e 9 999), o que criaria falsos repetidos.

In [21]:
rejected_clean, corrections = cleaner.clean(
    PickleStore.load(config.REJECTED_DEV_RAW_PKL, expected_type=pd.DataFrame), log)

# As exclusões são as mesmas das secções 11 e 12
assert log.steps[-2].rows_removed == (rejected_dev["policy_code"] == NEW_PRODUCTS_POLICY_CODE).sum()
assert log.steps[-1].rows_removed == repeated.sum()

display(log.to_frame())
corrections.to_frame()

,step,kind,rule,reason,rows_before,rows_removed,rows_after
n,,,,,,,
1,Pedidos anteriores ao primeiro aceite,partição,Application Date em 2007-05,"O primeiro empréstimo aceite é de 2007-06: sem aceites no mesmo mês, não há comparação",27648741,49,27648692
2,Separar o teste final,partição,Application Date de 2015-01 a 2015-12,Mesmo corte dos aceites; os rejeitados do teste não são usados (não entram nos modelos),27648692,2859379,24789313
3,Separar o período reservado,partição,Application Date de 2016-01 a 2018-12,Não usado na Parte 1 (mesmo corte dos aceites),24789313,21339229,3450084
4,Produtos novos não públicos,remoção,policy_code = 2,Os aceites são todos policy_code 1: comparar só pedidos dos mesmos produtos,3450084,6241,3443843
5,Pedidos repetidos,remoção,linha igual a uma anterior em todas as colunas (fica a primeira),O mesmo pedido submetido mais de uma vez no mesmo dia; contaria como várias rejeições,3443843,19615,3424228


,corrigidos
risk_score = 0,86038
"dti = -1, 9999, 99999, 199998",264300
dti > 1000 (sem os códigos),8253
amount_requested = 0,2


In [22]:
# Resultado: % em falta depois da limpeza e pedidos de cada escala do risk_score
display((rejected_clean.isna().mean() * 100).round(2).rename("% em falta").to_frame())
pd.crosstab(rejected_clean["risk_score_scale"], rejected_clean["application_date"].dt.year.rename("ano"))

,% em falta
amount_requested,0.00
application_date,0.00
title,0.45
risk_score,10.85
dti,7.96
zip_code,0.00
addr_state,0.00
emp_length,1.48
policy_code,0.00
risk_score_scale,0.00


ano,2007,2008,2009,2010,2011,2012,2013,2014
risk_score_scale,,,,,,,,
FICO,5223,25590,56960,112316,217325,336649,648126,0
VantageScore,0,0,0,0,0,0,109731,1912308


In [23]:
PickleStore.save(rejected_clean, config.REJECTED_DEV_PKL)
PickleStore.save(log, config.REJECTED_DEV_LOG_PKL)

out = write_exclusions_markdown(
    [log], config.DOCS_DIR / "02_preprocessamento" / "tabela_exclusoes_rejeitados.md",
    title="Tabela de exclusões: pedidos rejeitados", notebook="01b_rejeitados.ipynb")

print(f"Rejeitados do desenvolvimento limpos: {len(rejected_clean):,} linhas x "
      f"{rejected_clean.shape[1]} colunas -> {config.REJECTED_DEV_PKL.name}")
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

Rejeitados do desenvolvimento limpos: 3,424,228 linhas x 10 colunas -> rejeitados_dev.pkl
Escrito: docs\02_preprocessamento\tabela_exclusoes_rejeitados.md
